<a href="https://colab.research.google.com/github/missingstuffedbun/FloodEvacPlanner/blob/main/llm_beijing/Beijing1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prerequisite

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

data_path = os.path.join(os.path.curdir, "drive", "MyDrive", "ColabDatasets", "Beijing")
data_path

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
AMAP_KEY = userdata.get('AMAP_KEY')
WANDB_KEY = userdata.get('WANDB_KEY')

In [ ]:
import os
import wandb

os.environ["WANDB_API_KEY"] = WANDB_KEY
wandb.init(project="Beijing", entity="missingstuffedbun")

In [ ]:
HF_USERNAME = 'missingstuffedbun'

# Data preprocessing

In [ ]:
import pandas as pd
import numpy as np

file_path = "北京洪水 图文、视频数据共4,258条.xlsx"
# 读取第一个 sheet
df_text = pd.read_excel(os.path.join(data_path, file_path), sheet_name=0)
# 读取第二个 sheet
df_video = pd.read_excel(os.path.join(data_path, file_path), sheet_name=1)

# 简单检查
print("图文数据：", df_text.shape)
print("视频数据：", df_video.shape)


In [ ]:
df_text = df_text.rename(columns={
    "序号": "uid",
    "标题/微博内容": "title_weibo",
    "摘要": "summary",
    "原文/评论链接": "original_link",
    "来源网站": "source",
    "原文作者": "author",
    "日期": "date",
    "媒体类型": "media_type",
    "信息属性": "info_attribute",
    "信源地域": "source_region",
    "精准地域": "precise_region",
    "行业标签": "industry_tag",
    "涉及词": "related_words",
    "账号类型": "account_type",
    "主页链接": "homepage_link",
    "转发数": "repost_count",
    "评论数": "comment_count",
    "点赞数": "like_count",
    "粉丝数": "followers_count",
    "阅读数/浏览热度": "view_count",
    "收藏数": "bookmark_count",
    "标签_相关性": "tag_relevance",
    "标签_情绪": "tag_sentiment",
    "标签_意图": "tag_intent"
})

In [ ]:
df_video = df_video.rename(columns={
    "标题内容": "title_content",
    "摘要": "summary",
    "视频链接": "video_link",
    "视频来源": "source",
    "原文作者": "author",
    "发布日期": "date",
    "媒体类型": "media_type",
    "信息属性": "info_attribute",
    "信源地域": "source_region",
    "精准地域": "precise_region",
    "行业标签": "industry_tag",
    "涉及词": "related_words",
    "互动数": "interaction_count",
    "转发数": "repost_count",
    "点赞数": "like_count",
    "评论数": "comment_count",
    "粉丝数": "followers_count",
    "视频时长": "video_duration",
    "封面内容": "cover_content",
    "字幕内容": "subtitle_content",
    "背景内容": "background_content",
    "音频内容": "audio_content",
    "标签_相关性": "tag_relevance",
    "标签_情绪": "tag_sentiment",
    "标签_意图": "tag_intent"
})


In [ ]:
df_video['uid'] = range(1, len(df_video) + 1)

In [ ]:
# 将 "无" 替换为 NaN
text_columns = ['title_content', 'summary', 'cover_content', 'subtitle_content']
df_video[text_columns] = df_video[text_columns].replace('无', np.nan)

In [ ]:
import re
import numpy as np
import pandas as pd
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

# 固定 tokenizer
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-roberta-wwm-ext")



def extract_summary(df, text_columns_weights, max_tokens=500, sim_threshold=0.7, tfidf_max_features=2000):
    text_columns = list(text_columns_weights.keys())

    # 替换 "无" 为 NaN
    df[text_columns] = df[text_columns].replace('无', np.nan)

    # 分词
    def tokenize(text):
        return " ".join(jieba.lcut(text))

    # 按句拆分
    def split_sentences(text):
        sentences = re.split(r'(?<=[。！？\?])', text)
        return [s.strip() for s in sentences if s.strip()]

    # 生成每行句子列表及权重
    df['weighted_sentences'] = df.apply(
        lambda row: [
            (sentence, text_columns_weights[col])
            for col in text_columns if pd.notna(row[col])
            for sentence in split_sentences(row[col])
        ],
        axis=1
    )

    # TF-IDF 向量化所有候选句子
    all_sentences = []
    for sents in df['weighted_sentences']:
        for sent, w in sents:
            all_sentences.append(sent)
    all_sentences = list(dict.fromkeys(all_sentences))  # 去重完全相同句

    vectorizer = TfidfVectorizer(max_features=tfidf_max_features)
    vectorizer.fit(all_sentences)

    # 生成摘要
    summaries = []

    for sents_with_weight in df['weighted_sentences']:
        if not sents_with_weight:
            summaries.append('')
            continue

        sentence_texts = [s for s, w in sents_with_weight]
        sentence_weights = [w for s, w in sents_with_weight]

        sent_vecs = vectorizer.transform(sentence_texts)
        sim_matrix = cosine_similarity(sent_vecs)

        # 每句重要性 = TF-IDF sum * 权重
        scores = np.array(sent_vecs.sum(axis=1)).flatten() * np.array(sentence_weights)
        ranked_idx = np.argsort(scores)[::-1]

        summary_sentences = []
        token_count = 0
        selected_idx = []

        for idx in ranked_idx:
            # 与已选句子比较相似度
            if selected_idx:
                sim_to_selected = [sim_matrix[idx, j] for j in selected_idx]
                if max(sim_to_selected) > sim_threshold:
                    continue

            sent = sentence_texts[idx]

            # 使用 tokenizer 计算 token 数量
            sent_tokens = tokenizer.encode(sent, add_special_tokens=False)
            token_count += len(sent_tokens)

            if token_count > max_tokens:
                break

            summary_sentences.append(sent)
            selected_idx.append(idx)

        summaries.append(''.join(summary_sentences))

    df['concat_text'] = summaries
    return df

In [ ]:
weights_video = {
    'title_content': 3.0,
    'summary': 4.0,
    'cover_content': 2.0,
    'subtitle_content': 1.0,
    'background_content': 1.0
}

df_video = extract_summary(df_video, weights_video, max_tokens=500, sim_threshold=0.7)

weights_text = {
    'title_weibo': 2.0,
    'summary': 4.0
}

df_text = extract_summary(df_text, weights_text, max_tokens=500, sim_threshold=0.7)

In [ ]:
df_text['text_video'] = 'text'
df_video['text_video'] = 'video'

In [ ]:
# 要保留的列
columns_to_keep = ['uid', 'concat_text', 'tag_relevance', 'tag_sentiment', 'tag_intent', 'precise_region', 'source_region', 'source', 'text_video']

# 按行拼接
df_concat = pd.concat([df_text[columns_to_keep], df_video[columns_to_keep]], axis=0, ignore_index=True)

In [ ]:
tag_cols = ['tag_relevance', 'tag_sentiment', 'tag_intent']

# 先把 "无" 或空字符串统一为 NaN
df_concat[tag_cols] = df_concat[tag_cols].replace("无", pd.NA).replace("", pd.NA)

# 筛选三列都非空的行
df_concat = df_concat.dropna(subset=tag_cols, how='any')  # any 表示只要有 NaN 就删除

In [ ]:
import ast

df_concat['tag_intent'] = df_concat['tag_intent'].apply(
    lambda s: s[s.find('{') : s.rfind('}')+1] if isinstance(s, str) and '{' in s and '}' in s else s
)

In [ ]:
# 1️⃣ tag_relevance 映射
relevance_map = {0: "该消息与洪灾无关", 1: "该消息与洪灾相关"}
df_concat['tag_text_relevance'] = df_concat['tag_relevance'].astype(int).map(relevance_map)

# 2️⃣ tag_sentiment 映射
sentiment_map = {
    "positive": "表达的情绪为积极",
    "neutral": "表达的情绪为中立",
    "negative": "表达的情绪为消极"
}
df_concat['tag_text_sentiment'] = df_concat['tag_sentiment'].map(sentiment_map)

# 3️⃣ tag_intent 映射
intent_map = {
    'Information provided': '提供信息',
    'Sentiment and personal expression': '情绪表达',
    'Commentary and evaluation': '评价与建议',
    'Request for help': '求助信息',
    'aid coordination': '互助信息',
    'Non-informative': '无效信息'
}

def convert_intent(dict_string):
    try:
        # 将字符串转换为 dict
        d = ast.literal_eval(dict_string)
        # 只保留 keys
        keys = d.keys()
        # 映射成中文
        return [intent_map.get(k, k) for k in keys]
    except:
        return []

df_concat['tag_text_intent'] = df_concat['tag_intent'].apply(convert_intent)

In [ ]:
df_concat.to_csv(os.path.join(data_path, 'Beijing.csv'), index=False, encoding='utf-8-sig')

# Load data

In [ ]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(data_path, 'Beijing.csv'), encoding='utf-8-sig')

In [ ]:
df = df.dropna(subset=['concat_text'])
df['concat_text'] = df['concat_text'].astype(str)

In [ ]:
df.head()

# Functions

## Random samples

In [ ]:
import random
from collections import Counter

def samples_balanced(texts, labels, seed=202507):
    """
    随机抽取每个标签的 few-shot 示例，并按照标签在数据中的数量排序（多的在前）

    texts: list[str]
    labels: list[int] 或 list[str]

    返回:
        examples: [(text, label_str), ...]  # 每个标签一个示例，标签转为字符串
        candidate_labels: [label_str, ...]  # 按出现次数从多到少排序
    """
    random.seed(seed)
    examples = []

    # 统计标签频率
    label_counts = Counter(labels)
    # 按数量降序排列
    sorted_labels = [label for label, _ in label_counts.most_common()]
    candidate_labels = [l for l in sorted_labels]

    for l in sorted_labels:
        indices = [i for i, lab in enumerate(labels) if lab == l]
        sampled_index = random.choice(indices)  # 每个标签抽一个示例
        examples.append((texts[sampled_index], str(l)))  # 转为字符串

    return examples, candidate_labels


In [ ]:
import random
from collections import Counter

def samples_balanced_list(texts, labels_list, seed=202507):
    """
    随机抽取每个标签的 few-shot 示例，并按照标签在数据中的数量排序（多的在前）

    texts: list[str]             # 所有文本
    labels_list: list[list[str]] # 每条文本对应一个标签列表，例如 ["意图A", "意图B"]

    返回:
        examples: [(text, label_str), ...]  # 每个标签抽取一个示例
        candidate_labels: [label_str, ...]  # 按出现次数从多到少排序
    """
    random.seed(seed)
    examples = []

    # 统计每个标签出现次数
    all_labels = [label for sublist in labels_list for label in sublist]
    label_counts = Counter(all_labels)

    # 按数量降序排列
    sorted_labels = [label for label, _ in label_counts.most_common()]
    candidate_labels = sorted_labels.copy()

    for l in sorted_labels:
        # 找出包含该标签的文本索引
        indices = [i for i, lbls in enumerate(labels_list) if l in lbls]
        if indices:
            sampled_index = random.choice(indices)
            examples.append((texts[sampled_index], l))

    return examples, candidate_labels


In [ ]:
import random
from collections import Counter

def samples_balanced_dict(texts, labels_dict, seed=202507):
    """
    随机抽取每个标签的 few-shot 示例，并按照标签在数据中的数量排序（多的在前）

    texts: list[str]                   # 所有文本
    labels_dict: list[dict]             # 每条文本对应一个 dict，例如 {"意图A": "文本A", "意图B": "文本B"}

    返回:
        examples: [(text, label_str), ...]  # 每个标签抽取一个示例，标签转为字符串
        candidate_labels: [label_str, ...]  # 按出现次数从多到少排序
    """
    random.seed(seed)
    examples = []

    # 统计每个标签出现次数
    all_labels = []
    for d in labels_dict:
        all_labels.extend(d.keys())  # 只统计 dict 的 key（标签）
    label_counts = Counter(all_labels)

    # 按数量降序排列
    sorted_labels = [label for label, _ in label_counts.most_common()]
    candidate_labels = sorted_labels.copy()

    for l in sorted_labels:
        # 找出包含该标签的文本索引
        indices = [i for i, d in enumerate(labels_dict) if l in d]
        if indices:
            sampled_index = random.choice(indices)
            examples.append((texts[sampled_index], str(l)))

    return examples, candidate_labels


## Classification: single label

#### Zero-shot

In [ ]:
from transformers import pipeline, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

def predict_zeroshot(model_name, texts, labels, candidate_labels,
                     device=0, batch_size=32, max_length=512):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # 彻底锁死 tokenizer 最大长度
    tokenizer.model_max_length = max_length

    label2id = {label: idx for idx, label in enumerate(candidate_labels)}
    true_ids = [label2id[l] for l in labels]

    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=tokenizer,
        device=device,
        truncation=False,     # 禁止 pipeline 自己截断（这是关键！）
    )

    # ----------- 关键：严格截断 -----------
    def ultra_safe_truncate(text):
        encoded = tokenizer.encode(
            text,
            add_special_tokens=True
        )

        # 如果太长 → 强制裁剪到 max_length
        if len(encoded) > max_length:
            encoded = encoded[:max_length]

        # 再 decode，严格保持 token 不变
        return tokenizer.decode(
            encoded,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False
        )

    texts = [ultra_safe_truncate(t) for t in texts]

    pred_ids, probs = [], []

    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        batch_texts = texts[i:i+batch_size]

        outputs = classifier(
            batch_texts,
            candidate_labels=candidate_labels
        )

        for output in outputs:
            top_label = output["labels"][0]
            score = output["scores"][0]

            pred_ids.append(label2id[top_label])
            probs.append(score)

    acc = accuracy_score(true_ids, pred_ids)
    f1 = f1_score(true_ids, pred_ids, average="weighted")

    print(f"Zero-shot Accuracy: {acc:.4f}, F1: {f1:.4f}")

    return pred_ids, probs, {"accuracy": acc, "f1": f1}


### Few-shot

In [ ]:
from transformers import AutoTokenizer, pipeline
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm


def predict_fewshot(
    model_name,
    texts,
    labels,
    candidate_labels,
    examples,
    device=-1,
    batch_size=32,
    max_token_length=512
):
    """
    Few-shot 分类（单标签）。
    使用 tokenizer 控制长度，防止超过模型最大输入。
    每个 example 单独截断，example block < max_token_length / 2
    文本 block < max_token_length / 2
    最终 prompt 严格 < max_token_length
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.model_max_length = max_token_length

    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=tokenizer,
        multi_label=False,
        device=device
    )

    label2id = {label: idx for idx, label in enumerate(candidate_labels)}
    true_ids = [label2id[l] for l in labels]

    # ---------------- 构建 example block ----------------
    half = max_token_length // 2
    n_examples = max(len(examples), 1)
    per_example_limit = half // n_examples - 5  # 每个 example 最大 token 数

    example_parts = []
    used_tokens = 0
    for t, l in examples:
        raw = f"文本：{t}\n判断：{l}\n\n"
        ids = tokenizer(raw, add_special_tokens=False)["input_ids"]

        # 每个 example 截断
        if len(ids) > per_example_limit:
            ids = ids[:per_example_limit]
            raw = tokenizer.decode(ids, skip_special_tokens=True)

        if used_tokens + len(ids) > half:
            break

        example_parts.append(raw)
        used_tokens += len(ids)

    example_block = "示例：\n" + "".join(example_parts)

    # ---------------- 构建最终 prompt ----------------
    max_text_tokens = half
    prompt_texts = []

    for t in texts:
        text_ids = tokenizer(t, add_special_tokens=False)["input_ids"]
        if len(text_ids) > max_text_tokens:
            text_ids = text_ids[:max_text_tokens]
        t_trunc = tokenizer.decode(text_ids, skip_special_tokens=True)

        prompt = f"{example_block}文本：{t_trunc}\n判断："

        # 最终严格检查
        final_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        if len(final_ids) > max_token_length:
            final_ids = final_ids[:max_token_length]
            prompt = tokenizer.decode(final_ids, skip_special_tokens=True)

        prompt_texts.append(prompt)

    # ---------------- 批量推理 ----------------
    pred_ids, probs = [], []
    for i in tqdm(range(0, len(prompt_texts), batch_size)):
        batch_texts = prompt_texts[i:i+batch_size]
        outputs = classifier(
            batch_texts,
            candidate_labels=candidate_labels,
            truncation=True,         # 自动截断超长输入
            max_length=max_token_length
        )
        for output in outputs:
            top_label = output["labels"][0]
            score = output["scores"][0]
            pred_ids.append(label2id[top_label])
            probs.append(score)

    acc = accuracy_score(true_ids, pred_ids)
    f1 = f1_score(true_ids, pred_ids, average="weighted")
    print(f"Few-shot Accuracy: {acc:.4f}, F1: {f1:.4f}")

    return pred_ids, probs, {"accuracy": acc, "f1": f1}


### Fine-tune

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
import torch
import os
import shutil
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import HfApi, HfFolder, create_repo

def predict_finetune(
    model_name,
    texts,
    labels,
    candidate_labels,
    epochs=3,
    batch_size=8,
    max_length=512,
    device=0
):
    """
    Fine-tune 分类 + 计算准确率和 F1
    使用 tokenizer 进行 token 截断，避免长度过长
    labels 可以是原始字符串
    """
    # ---------------- 构建 label2id / id2label ----------------
    label2id = {label: idx for idx, label in enumerate(candidate_labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    # ---------------- 转换标签为整数 ----------------
    labels_id = [label2id[l] for l in labels]

    # ---------------- tokenizer & model ----------------
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(candidate_labels),
        ignore_mismatched_sizes=True
    )
    model.config.label2id = label2id
    model.config.id2label = id2label

    # ---------------- 构建 Dataset ----------------
    df = Dataset.from_pandas(pd.DataFrame({'text': texts, 'label': labels_id}))

    def tokenize_fn(examples):
        # 用 tokenizer 进行 token 截断
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )

    dataset = df.map(tokenize_fn, batched=True)
    dataset = dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset, val_dataset = dataset['train'], dataset['test']

    # ---------------- 训练参数 ----------------
    model_dir = f"./{model_name.replace('/', '_')}_finetuned"
    os.makedirs(model_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=model_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=1,
        do_eval=True,
        eval_steps=50,
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=lambda p: {
            "accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1)),
            "f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="weighted")
        }
    )

    # ---------------- 训练 ----------------
    trainer.train()

    # ---------------- 保存本地 ----------------
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"模型已保存到本地: {model_dir}")

    # ---------------- 上传 Hugging Face ----------------
    # 可选，如果没有 HF_TOKEN / HF_USERNAME，可以跳过
    try:
        hf_api = HfApi()
        HfFolder.save_token(HF_TOKEN)
        hf_repo_name = f"{HF_USERNAME}/{model_name.replace('/', '_')}_finetuned"
        create_repo(repo_id=hf_repo_name, token=HF_TOKEN, exist_ok=True, private=False)
        hf_api.upload_folder(
            folder_path=model_dir,
            path_in_repo=".",
            repo_id=hf_repo_name,
            repo_type="model",
            commit_message="Upload fine-tuned model",
            token=HF_TOKEN
        )
        print(f"模型已上传到 Hugging Face: {hf_repo_name}")
    except Exception as e:
        print(f"上传模型出错: {e}")

    # ---------------- 推理 ----------------
    classifier = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        max_length=max_length,
        return_all_scores=True,
        device=device if torch.cuda.is_available() else -1
    )

    pred_ids, probs = [], []
    for text in texts:
        # 使用 tokenizer 截断 token
        tokens = tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=max_length)
        text_trunc = tokenizer.decode(tokens)
        output = classifier(text_trunc)[0]
        top_item = max(output, key=lambda x: x['score'])
        # 转成 candidate_labels 对应的索引
        label_name = top_item['label']
        if label_name.startswith("LABEL_"):
            top_index = int(label_name.split("_")[1])
        else:
            top_index = label2id[label_name]
        pred_ids.append(top_index)
        probs.append(top_item['score'])

    # ---------------- 删除本地模型 ----------------
    shutil.rmtree(model_dir)
    print(f"本地模型已删除: {model_dir}")

    # ---------------- 返回指标 ----------------
    acc = accuracy_score(labels_id, pred_ids)
    f1 = f1_score(labels_id, pred_ids, average="weighted")
    print(f"Fine-tune Accuracy: {acc:.4f}, F1: {f1:.4f}")

    return pred_ids, probs, {"accuracy": acc, "f1": f1}


## Classification: multi label

### Few-shot

In [ ]:
from transformers import AutoTokenizer, pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, accuracy_score
from tqdm import tqdm

def predict_fewshot_multilabel(
    model_name,
    texts,
    labels,
    candidate_labels,
    examples,
    device=-1,
    batch_size=16,
    max_token_length=512,
    threshold=0.5
):
    """
    Few-shot 多标签分类（自动控制 token 长度）
    使用 tokenizer 控制长度，防止超过模型最大输入。
    每个 example 单独截断，example block < max_token_length / 2
    文本 block < max_token_length / 2
    最终 prompt 严格 < max_token_length
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.model_max_length = max_token_length

    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        tokenizer=tokenizer,
        multi_label=True,
        device=device
    )

    label2id = {label: idx for idx, label in enumerate(candidate_labels)}
    mlb = MultiLabelBinarizer(classes=candidate_labels)
    true_bin = mlb.fit_transform(labels)

    # ---------------- 构建 example block ----------------
    half = max_token_length // 2
    n_examples = max(len(examples), 1)
    per_example_limit = half // n_examples - 5  # 每个 example 最大 token 数

    example_parts = []
    used_tokens = 0
    for t, lab in examples:
        raw = f"文本：{t}\n判断：{', '.join(lab)}\n\n"
        ids = tokenizer(raw, add_special_tokens=False)["input_ids"]

        if len(ids) > per_example_limit:
            ids = ids[:per_example_limit]
            raw = tokenizer.decode(ids, skip_special_tokens=True)

        if used_tokens + len(ids) > half:
            break

        example_parts.append(raw)
        used_tokens += len(ids)

    example_block = "示例：\n" + "".join(example_parts)

    # ---------------- 构建最终 prompt ----------------
    max_text_tokens = half
    prompt_texts = []
    for t in texts:
        text_ids = tokenizer(t, add_special_tokens=False)["input_ids"]
        if len(text_ids) > max_text_tokens:
            text_ids = text_ids[:max_text_tokens]
        t_trunc = tokenizer.decode(text_ids, skip_special_tokens=True)

        prompt = f"{example_block}文本：{t_trunc}\n判断："

        final_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        if len(final_ids) > max_token_length:
            final_ids = final_ids[:max_token_length]
            prompt = tokenizer.decode(final_ids, skip_special_tokens=True)

        prompt_texts.append(prompt)

    # ---------------- 批量推理 ----------------
    pred_ids, probs = [], []
    for i in tqdm(range(0, len(prompt_texts), batch_size)):
        batch_texts = prompt_texts[i:i + batch_size]
        outputs = classifier(batch_texts, candidate_labels=candidate_labels, truncation=True, max_length=max_token_length)
        for output in outputs:
            pred_labels, pred_scores = [], []
            for label, score in zip(output['labels'], output['scores']):
                if score >= threshold:
                    pred_labels.append(label2id[label])
                    pred_scores.append(score)
            pred_ids.append(pred_labels)
            probs.append(pred_scores)

    # ---------------- 指标计算 ----------------
    pred_bin = mlb.transform([[candidate_labels[i] for i in p] for p in pred_ids])

    metrics = {}
    # Strict exact-match accuracy
    metrics['exact_accuracy'] = (pred_bin == true_bin).all(axis=1).mean()
    # Hamming loss
    metrics['hamming_loss'] = hamming_loss(true_bin, pred_bin)
    # Micro metrics
    metrics['micro_f1'] = f1_score(true_bin, pred_bin, average="micro")
    metrics['micro_precision'] = precision_score(true_bin, pred_bin, average="micro")
    metrics['micro_recall'] = recall_score(true_bin, pred_bin, average="micro")
    # Macro metrics
    metrics['macro_f1'] = f1_score(true_bin, pred_bin, average="macro")
    metrics['macro_precision'] = precision_score(true_bin, pred_bin, average="macro")
    metrics['macro_recall'] = recall_score(true_bin, pred_bin, average="macro")

    # 打印
    print("Few-shot multi-label metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    return pred_ids, probs, metrics


### Fine-tune

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
import torch
import os
import shutil
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, accuracy_score
from huggingface_hub import HfApi, HfFolder, create_repo

def predict_finetune_multilabel(
    model_name,
    texts,
    labels,
    candidate_labels,
    epochs=3,
    batch_size=8,
    max_length=512,
    device=0,
    hf_token=None,
    hf_username=None
):
    """
    Fine-tune 多标签分类 + 本地保存/删除模型 + pipeline 推理 + 上传 HF
    labels: list of list[str]
    """
    # ---------------- 1. MultiLabelBinarizer ----------------
    mlb = MultiLabelBinarizer(classes=candidate_labels)
    labels_bin = mlb.fit_transform(labels).astype(float)

    # ---------------- 2. label2id / id2label ----------------
    label2id = {label: idx for idx, label in enumerate(candidate_labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    # ---------------- 3. Tokenizer & Model ----------------
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(candidate_labels),
        problem_type="multi_label_classification"
    )
    model.config.label2id = label2id
    model.config.id2label = id2label

    # ---------------- 4. Dataset ----------------
    df = Dataset.from_pandas(pd.DataFrame({'text': texts, 'labels': list(labels_bin)}))

    def tokenize_fn(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )

    dataset = df.map(tokenize_fn, batched=True)
    dataset = dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset, val_dataset = dataset['train'], dataset['test']

    # ---------------- 5. TrainingArguments & Trainer ----------------
    model_dir = f"./{model_name.replace('/', '_')}_finetuned_multilabel"
    os.makedirs(model_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=model_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=1,
        do_eval=True,
        eval_steps=50,
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=False
    )

    def compute_metrics(p):
        preds = torch.sigmoid(torch.tensor(p.predictions))
        preds_bin = (preds >= 0.5).int().numpy()
        # 这里只返回微平均 F1 供 trainer 参考
        return {"micro_f1": f1_score(p.label_ids, preds_bin, average="micro")}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # ---------------- 6. Train ----------------
    trainer.train()

    # ---------------- 7. Save local model ----------------
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"模型已保存到本地: {model_dir}")

    # ---------------- 8. Upload Hugging Face ----------------
    if hf_token is not None and hf_username is not None:
        try:
            hf_api = HfApi()
            HfFolder.save_token(hf_token)
            hf_repo_name = f"{hf_username}/{model_name.replace('/', '_')}_finetuned_multilabel"
            create_repo(repo_id=hf_repo_name, token=hf_token, exist_ok=True, private=False)
            hf_api.upload_folder(
                folder_path=model_dir,
                path_in_repo=".",
                repo_id=hf_repo_name,
                repo_type="model",
                commit_message="Upload fine-tuned multi-label model",
                token=hf_token
            )
            print(f"模型已上传到 Hugging Face: {hf_repo_name}")
        except Exception as e:
            print(f"上传模型出错: {e}")

    # ---------------- 9. 推理 pipeline ----------------
    classifier = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device if torch.cuda.is_available() else -1,
        truncation=True,
        max_length=max_length,
        return_all_scores=True
    )

    pred_ids, probs = [], []
    for text in texts:
        output = classifier(text)[0]
        pred_labels, pred_scores = [], []
        for item in output:
            if item['score'] >= 0.5:
                pred_labels.append(label2id[item['label']])
                pred_scores.append(item['score'])
        pred_ids.append(pred_labels)
        probs.append(pred_scores)

    # ---------------- 10. 删除本地模型 ----------------
    shutil.rmtree(model_dir)
    print(f"本地模型已删除: {model_dir}")

    # ---------------- 11. Metrics ----------------
    pred_bin = mlb.transform([[candidate_labels[i] for i in p] for p in pred_ids])

    metrics = {}
    # Strict exact-match accuracy
    metrics['exact_accuracy'] = (pred_bin == labels_bin).all(axis=1).mean()
    # Hamming loss
    metrics['hamming_loss'] = hamming_loss(labels_bin, pred_bin)
    # Micro metrics
    metrics['micro_f1'] = f1_score(labels_bin, pred_bin, average="micro")
    metrics['micro_precision'] = precision_score(labels_bin, pred_bin, average="micro")
    metrics['micro_recall'] = recall_score(labels_bin, pred_bin, average="micro")
    # Macro metrics
    metrics['macro_f1'] = f1_score(labels_bin, pred_bin, average="macro")
    metrics['macro_precision'] = precision_score(labels_bin, pred_bin, average="macro")
    metrics['macro_recall'] = recall_score(labels_bin, pred_bin, average="macro")

    # 打印
    print("Multi-label metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    return pred_ids, probs, metrics


## Run models

In [ ]:
def run_models_from_dict(
    model_modes_dict,
    texts,
    labels,
    candidate_labels,
    examples,
    device=-1
):
    """
    从一个 dict 运行多个模型，每个模型可配置不同模式。

    参数:
        model_modes_dict: dict, 结构为 {model_name: [modes]}
        texts: list[str], 文本
        labels: list[int], 标签
        candidate_labels: list[str], 分类标签
        examples: few-shot 示例 (list of (text, label))，可选
        device: int, 运行设备（-1=CPU, >=0 为GPU）
    """
    all_results = {}

    for model_name, modes in model_modes_dict.items():
        print(f"\n==============================")
        print(f"🚀 正在运行模型: {model_name}")
        print(f"模式: {modes}")
        print(f"==============================")

        results_summary = {}

        # --- Zero-shot ---
        if "zero-shot" in modes:
            print(f"\n===== Running Zero-shot: {model_name} =====")
            res, probs, metrics = predict_zeroshot(
                model_name=model_name,
                texts=texts,
                labels=labels,
                candidate_labels=candidate_labels,
                device=device
            )
            results_summary["zero-shot"] = {"metrics": metrics, "preds": res, "probs": probs}

        # --- Few-shot ---
        if "few-shot" in modes:
            print(f"\n===== Running Few-shot: {model_name} =====")
            res, probs, metrics = predict_fewshot(
                model_name=model_name,
                texts=texts,
                labels=labels,
                candidate_labels=candidate_labels,
                examples=examples,
                device=device
            )
            results_summary["few-shot"] = {"metrics": metrics, "preds": res, "probs": probs}

        # --- Fine-tune ---
        if "fine-tune" in modes:
            print(f"\n===== Running Fine-tune: {model_name} =====")
            res, probs, metrics = predict_finetune(
                model_name=model_name,
                texts=texts,
                labels=labels,
                candidate_labels=candidate_labels,
                device=device
            )
            results_summary["fine-tune"] = {"metrics": metrics, "preds": res, "probs": probs}

        # --- 汇总输出 ---
        all_results[model_name] = results_summary

    print("\n🎯 所有模型运行结束！")
    return all_results


In [ ]:
def run_models_from_dict_multilabel(
    model_modes_dict,
    texts,
    labels,
    candidate_labels,
    examples=None,
    device=-1
):
    """
    从一个 dict 运行多个模型，每个模型可配置不同模式。
    支持多标签（list of list[str]）分类。

    参数:
        model_modes_dict: dict, 结构为 {model_name: [modes]}
        texts: list[str], 文本
        labels: list of list[str], 多标签
        candidate_labels: list[str], 分类标签
        examples: few-shot 示例 (list of (text, list[str]))，可选
        device: int, 运行设备（-1=CPU, >=0 为GPU）
    """
    all_results = {}

    for model_name, modes in model_modes_dict.items():
        print(f"\n==============================")
        print(f"🚀 正在运行模型: {model_name}")
        print(f"模式: {modes}")
        print(f"==============================")

        results_summary = {}

        # --- Zero-shot multi-label ---
        if "zero-shot-multi" in modes:
            print(f"\n===== Running Zero-shot (multi-label): {model_name} =====")
            res, probs, metrics = predict_zero_shot_multilabel(
                model_name=model_name,
                texts=texts,
                labels=labels,
                candidate_labels=candidate_labels,
                device=device
            )
            results_summary["zero-shot_multi"] = {"metrics": metrics, "preds": res, "probs": probs}

        # --- Few-shot multi-label ---
        if "few-shot-multi" in modes:
            if examples is None:
                raise ValueError("Few-shot 模式需要提供 examples 参数")
            print(f"\n===== Running Few-shot (multi-label): {model_name} =====")
            res, probs, metrics = predict_fewshot_multilabel(
                model_name=model_name,
                texts=texts,
                labels=labels,
                candidate_labels=candidate_labels,
                examples=examples,
                device=device
            )
            results_summary["few-shot_multi"] = {"metrics": metrics, "preds": res, "probs": probs}

        # --- Fine-tune multi-label ---
        if "fine-tune-multi" in modes:
            print(f"\n===== Running Fine-tune (multi-label): {model_name} =====")
            res, probs, metrics = predict_finetune_multilabel(
                model_name=model_name,
                texts=texts,
                labels=labels,
                candidate_labels=candidate_labels,
                device=device
            )
            results_summary["fine-tune_multi"] = {"metrics": metrics, "preds": res, "probs": probs}

        # --- 汇总输出 ---
        all_results[model_name] = results_summary

    print("\n🎯 所有模型运行结束！")
    return all_results


## Save results

In [ ]:
import os
import pandas as pd

def save_results(all_results, texts, labels, suffix="results"):
    """
    保存模型结果

    参数:
        all_results: dict, run_models_from_dict 的返回值
        texts: list[str], 原始文本
        labels: list[int], 原始标签
        suffix: str, 文件名后缀
    返回:
        metric_csv_path, preds_csv_path
    """
    os.makedirs(data_path, exist_ok=True)

    # ---------------- 保存 metrics ----------------
    metrics_list = []
    for model_name, modes_result in all_results.items():
        for mode, result in modes_result.items():
            metrics_row = {
                "model_name": model_name,
                "mode": mode
            }
            # 动态加入 metrics 中所有键值
            for metric_name, metric_value in result["metrics"].items():
                metrics_row[metric_name] = metric_value
            metrics_list.append(metrics_row)

    df_metrics = pd.DataFrame(metrics_list)
    metric_csv_path = os.path.join(data_path, f"metrics_{suffix}.csv")
    df_metrics.to_csv(metric_csv_path, index=False, encoding="utf-8-sig")
    print(f"模型指标已保存到: {metric_csv_path}")

    # ---------------- 保存 preds & probs ----------------
    df_preds = pd.DataFrame({
        "text": texts,
        "labels": labels
    })
    for model_name, modes_result in all_results.items():
        for mode, result in modes_result.items():
            tag = f"{model_name}_{mode}"
            df_preds[f"{tag}_preds"] = result["preds"]
            df_preds[f"{tag}_probs"] = result["probs"]

    preds_csv_path = os.path.join(data_path, f"predicts_{suffix}.csv")
    df_preds.to_csv(preds_csv_path, index=False, encoding="utf-8-sig")
    print(f"模型预测结果已保存到: {preds_csv_path}")

    return metric_csv_path, preds_csv_path


# Task: Relevance

In [ ]:
texts = df['concat_text'].tolist()
labels = df['tag_text_relevance'].tolist()

In [ ]:
examples, candidate_labels = samples_balanced(texts, labels)
print(examples, '\n', candidate_labels)

In [ ]:
model_modes = {
    # 中文模型：Few-shot & Fine-tune 效果好，Zero-shot 需要手动 prompt 设计
    "hfl/chinese-roberta-wwm-ext": ["few-shot", "fine-tune"],
    "bert-base-chinese": ["few-shot", "fine-tune"],
    "uer/roberta-base-finetuned-jd-binary-chinese": ["few-shot", "fine-tune"]
}


In [ ]:
results = run_models_from_dict(
    model_modes_dict=model_modes,
    texts=texts,
    labels=labels,
    candidate_labels=candidate_labels,
    examples=examples,
    device=0
)


In [ ]:
for model_name in results.keys():
  print(model_name, '\n', results.get(model_name))

In [ ]:
save_results(all_results=results, texts=texts, labels=labels, suffix="relevance")

# Task: Sentiment

In [ ]:
# texts = df['concat_text'] + '\n' + df['tag_text_relevance']
texts = df['concat_text']
labels = df['tag_text_sentiment']

In [ ]:
examples, candidate_labels = samples_balanced(texts, labels)
print(examples, '\n', candidate_labels)

In [ ]:
model_modes = {
    # "cardiffnlp/twitter-roberta-base-sentiment-latest": ["few-shot", "fine-tune"],
    # "jackietung/bert-base-chinese-finetuned-sentiment": ["few-shot", "fine-tune"],
    "cardiffnlp/twitter-roberta-base-sentiment-latest": ["zero-shot"],
    "jackietung/bert-base-chinese-finetuned-sentiment": ["zero-shot"],
}

In [ ]:
results = run_models_from_dict(
    model_modes_dict=model_modes,
    texts=texts,
    labels=labels,
    candidate_labels=candidate_labels,
    examples=examples,
    device=0
)

In [ ]:
save_results(all_results=results, texts=texts, labels=labels, suffix="sentiment-zero")

# Task: Intention

In [ ]:
import ast

texts = df['concat_text'] + '\n' + df['tag_text_relevance'] + '\n' + df['tag_text_sentiment']
labels = df['tag_text_intent'].apply(lambda s: ast.literal_eval(s)).tolist()  # 字符串形式

In [ ]:
examples, candidate_labels = samples_balanced_list(texts, labels)
print(examples, '\n', candidate_labels)

In [ ]:
model_modes = {
    # 中文通用模型
    "hfl/chinese-macbert-base": ["few-shot-multi", "fine-tune-multi"],       # MacBERT，短文本多标签表现好
    # 中文RoBERTa模型（效果一般，可对比）
    "hfl/chinese-roberta-wwm-ext": ["few-shot-multi", "fine-tune-multi"],   # 中文RoBERTa WWM
    # 多语言生成模型（可做文本生成型多标签预测）
    "google/mt5-small": ["few-shot-multi", "fine-tune-multi"],              # 小型mT5，可做prompt生成式few-shot
}


In [ ]:
results = run_models_from_dict_multilabel(
    model_modes_dict=model_modes,
    texts=texts,
    labels=labels,
    candidate_labels=candidate_labels,
    examples=examples,
    device=0
)


In [ ]:
save_results(all_results=results, texts=texts, labels=labels, suffix="intent")

# Task: Action insight (demised)


## Filtered posts

In [ ]:
import pandas as pd
import ast

def process_row(row):
    # 1. 忽略洪灾无关
    if row.get('tag_text_relevance') == '该消息与洪灾无关':
        return None
    # 2. 忽略情绪积极
    if row.get('tag_text_sentiment') == '表达的情绪为积极':
        return None
    # 3. 处理信息内容
    intents_str = row.get('tag_text_intent', '[]')
    try:
        intents = ast.literal_eval(intents_str) if isinstance(intents_str, str) else intents_str
    except:
        intents = []
    # 删除多标签中的无效信息
    intents = [i for i in intents if i not in ['情绪表达', '评价与建议', '无效信息']]
    if len(intents) == 0:
        return None
    row['tag_action'] = intents
    return row

# 使用 apply 并丢弃 None
df_filtered = df.apply(process_row, axis=1).dropna().reset_index(drop=True)
print("过滤后的数量:", len(df_filtered))


In [ ]:
df_filtered.head()

In [ ]:
texts = df_filtered['concat_text']
labels = df_filtered['tag_action']

## LLM modeling

In [ ]:
from transformers import pipeline
import pandas as pd
import torch

# 可换成你想要的小模型：
# "WeiboAI/VibeThinker-1.5B"
# "meta-llama/Llama-3.2-1B"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16,
    device=0 if torch.cuda.is_available() else -1
)


In [ ]:
from tqdm import tqdm

def generate_reports(texts, labels, batch_size=8, max_new_tokens=128):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        prompts = [
            f'''
            根据洪灾场景下的社交媒体文本，用简洁的语言，结合原文，给出建议采取的行动（例如：更新信息、寻求帮助等）。
            原文：{text}
            '''
            for text, label in zip(batch_texts, batch_labels)
        ]

        outputs = pipe(
            prompts,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

        # 去掉 prompt，只保留模型生成部分
        for o, prompt in zip(outputs, prompts):
            gen_text = o[0]["generated_text"]
            if gen_text.startswith(prompt):
                gen_text = gen_text[len(prompt):].strip()
            results.append(gen_text)

    return results


In [ ]:
reports = generate_reports(texts, labels)

In [ ]:
df_filtered['report'] = reports

In [ ]:
df_filtered.to_csv(os.path.join(data_path, 'Beijing_report.csv'), index=False, encoding='utf-8-sig')

# Task: Geolocation extraction

## LLM modeling

In [ ]:
from transformers import pipeline

# 加载地理命名实体识别模型（推荐）
model_name = "uer/roberta-base-finetuned-cluener2020-chinese"

# 初始化 pipeline
ner_pipeline = pipeline(
    "ner",
    model=model_name,
    tokenizer=model_name,
    token=HF_TOKEN  # ✅ 新写法，替代 use_auth_token
)

def geolocation_extraction(text):
  results = ner_pipeline(text)
  # 提取地理实体（可能模型不同，标签也不同，如 'LOC', 'GPE', 'address', 'ns' 等）
  geo_labels = {"LOC", "GPE", "ns", "address", "地名"}
  geo_entities = [r["word"] for r in results if r.get("entity") in geo_labels or "LOC" in r.get("entity", "")]
  # 拼接逻辑
  locations = []
  current = ""
  for r in results:
      if r["entity"].startswith("B-"):
          if current:
              locations.append(current)
          current = r["word"]
      elif r["entity"].startswith("I-") and current:
          current += r["word"]
      else:
          if current:
              locations.append(current)
              current = ""
  if current:
      locations.append(current)

  # 去重
  unique_locations = list(set(locations))
  return unique_locations

In [ ]:
df['ner'] = df['concat_text'].apply(geolocation_extraction)

In [ ]:
df["precise_region"] = df["precise_region"].str.split(r"[，,\s]+")
df["source_region"] = df["source_region"].str.split(r"[ ，,\s]+")

In [ ]:
df.head().T

In [ ]:
df['coords'] = df.apply(lambda row: dict(zip(row['ner'], geocode_amap.get(row['ner']))), axis=1))

## Amap coords extraction

In [ ]:
import requests


def geocode_amap(location, key=AMAP_KEY, city="北京"):
    """
    调用高德地理编码 API，并限定在指定城市
    返回 (经度, 纬度) 或 None
    """
    url = "https://restapi.amap.com/v3/place/text"
    params = {
        "keywords": location,
        "key": key,
        "city": city,
        "citylimit": True# 限定行政区
    }
    try:
        res = requests.get(url, params=params).json()
        if int(res.get('count'))>=1:
          return res.get("pois")[0]
        else:
          print(f"❌ 地址未找到或 geocodes 为空: {location}")
    except Exception as e:
        print(f"❌ 请求出错：{e}")
    return None


In [ ]:
# 所有 ner 的集合
ner_set = set(x for lst in df['ner'] for x in lst if x)

# 所有需要排除的地区
rm_set = set(x for lst in df['precise_region'] for x in lst if x) \
       | set(x for lst in df['source_region'] for x in lst if x)

# 需要 geocode 的 ner
to_geocode = ner_set - rm_set

# 调用 geocode_amap
coords_dict = {ner: geocode_amap(ner) for ner in to_geocode}


In [ ]:
len(coords_dict.keys())

In [ ]:
df['coords'] = df['ner'].apply(lambda lst: {ner: coords_dict[ner] for ner in lst if ner in coords_dict})

In [ ]:
df.to_csv(os.path.join(data_path, 'Beijing_coords.csv'), index=False, encoding='utf-8-sig')

## Mapping locations

In [ ]:
import json

# 保存到本地文件
with open(os.path.join(data_path, "coords.json"), "w", encoding="utf-8") as f:
    json.dump(coords_dict, f, ensure_ascii=False, indent=2)

In [ ]:
import json
import pandas as pd

with open(os.path.join(data_path, "coords.json"), "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for key, info in data.items():
    if info is None:
      continue
    name = info.get("name", None)
    loc = info.get("location", "")

    # location: "lat,lon"
    parts = [p.strip() for p in loc.split(",")]

    if len(parts) != 2:
        lat, lon = None, None
    else:
        lat = float(parts[0])
        lon = float(parts[1])

    rows.append({
        "key": key,
        "name": name,
        "lat": lat,
        "lon": lon
    })

df = pd.DataFrame(rows)
df.to_csv(os.path.join(data_path, "coords.csv"), index=False, encoding="utf-8-sig")